# 02-short. Colab용 LightGBM baseline 빠른 재현

기존 `02_lightgbm_baselines.ipynb`는 코드 검사용으로 길게 펼친 버전입니다.  
이 파일은 **Colab에서 짧게 실행**하기 위한 버전입니다.

- 한국어 주석을 많이 넣었습니다.
- 기본값은 빠른 데모를 위해 `LIMIT_SUBJECTS = 12`, `FOLDS = 3`입니다.
- 전체 재현은 `LIMIT_SUBJECTS = None`, `FOLDS = 5`로 바꾸면 됩니다.
- `DEVICE = "auto"`라서 LightGBM이 CUDA/OpenCL GPU를 먼저 시도하고 실패하면 CPU로 내려갑니다.


## 0. Colab/repo 환경 준비


In [ ]:
# Colab에서 이 노트북만 열어도 동작하도록 repo와 의존성을 준비합니다.
# 로컬 repo/Jupyter에서는 아무것도 설치하지 않고 현재 repo root만 찾습니다.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tlxhtls/llamac_research.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: Path | None = None) -> Path:
    """현재 위치에서 위로 올라가며 llamac_research repo root를 찾습니다."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "llamac_research").exists():
            return candidate
    raise RuntimeError(f"repo root를 찾지 못했습니다: {start}")


if IN_COLAB:
    root = Path("/content/llamac_research")
    if not (root / "src" / "llamac_research").exists():
        print("Colab에 repo를 clone합니다...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    os.chdir(root)
    print("Colab 의존성을 설치합니다. 처음 한 번은 1~3분 걸릴 수 있습니다...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".",
            "lightgbm>=4.5",
            "optuna>=4.0",
            "scikit-learn>=1.5",
            "xgboost>=2.1",
        ],
        check=True,
    )
    ROOT = root
else:
    ROOT = find_repo_root()
    os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT = {ROOT}")


## 1. 실험 설정


In [ ]:
# 빠르게 구조를 확인하려면 12명만 사용합니다. 전체 데이터는 None으로 바꾸세요.
LIMIT_SUBJECTS = 12
FOLDS = 3
SEED = 42
WORKERS = 2
DEVICE = "auto"  # auto = CUDA -> OpenCL GPU -> CPU 순서로 가능한 backend를 찾습니다.

# 이미 만든 feature/result를 재사용하면 Colab 런타임 재시작 후에도 빠릅니다.
REBUILD_FEATURES = False
RERUN_EXPERIMENTS = False

FEATURE_DIR = ROOT / "data" / "processed" / "short_colab"
RESULT_DIR = ROOT / "artifacts" / "short_colab" / "lightgbm_baselines"
FEATURE_ALL = FEATURE_DIR / "features_all_short.parquet"
FEATURE_PPG = FEATURE_DIR / "features_ppg_short.parquet"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f"LIMIT_SUBJECTS={LIMIT_SUBJECTS}, FOLDS={FOLDS}, DEVICE={DEVICE}")


## 2. 데이터셋 다운로드와 압축해제


In [ ]:
# Figshare DOI 10.6084/m9.figshare.28748696.v6 에서 zip을 내려받고 압축을 풉니다.
# 빠른 Colab 데모는 LIMIT_SUBJECTS명만 받습니다. 전체 재현은 LIMIT_SUBJECTS = None 으로 바꾸세요.
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import polars as pl

from scripts.download_llamac import (
    FigshareFile,
    build_dataset_index,
    download_one,
    extract_archives,
    fetch_metadata,
    natural_key,
    write_manifest,
)

RAW_DIR = ROOT / "data" / "raw"
EXTRACTED_DIR = ROOT / "data" / "extracted"
PROCESSED_DIR = ROOT / "data" / "processed"
INDEX_PATH = PROCESSED_DIR / "dataset_index.csv"
SKIP_HEAVY = os.environ.get("LLAMAC_SKIP_HEAVY") == "1"

ARTICLE_ID = "28748696"
VERSION = "6"
DOWNLOAD_WORKERS = 4
MAX_ZIPS = LIMIT_SUBJECTS  # None이면 전체 zip 파일을 다운로드합니다.

if INDEX_PATH.exists():
    print(f"이미 준비된 index가 있어서 다운로드를 건너뜁니다: {INDEX_PATH}")
elif SKIP_HEAVY:
    print("LLAMAC_SKIP_HEAVY=1 이므로 다운로드를 건너뜁니다.")
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    metadata = fetch_metadata(ARTICLE_ID, VERSION)
    files = [FigshareFile.from_api(item) for item in metadata["files"]]
    zip_files = sorted([f for f in files if f.name.lower().endswith(".zip")], key=lambda f: natural_key(f.name))
    selected = zip_files[:MAX_ZIPS] if MAX_ZIPS is not None else zip_files
    write_manifest(metadata, RAW_DIR, selected)

    print(f"다운로드 대상 zip: {len(selected)}개")
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_one, f, RAW_DIR, False, 3, 120, False) for f in selected]
        for i, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            print(f"[{i}/{len(selected)}] {result['name']}: {result['status']}")

    extract_archives(RAW_DIR, selected, EXTRACTED_DIR, force=False)
    build_dataset_index(EXTRACTED_DIR, INDEX_PATH)

if INDEX_PATH.exists():
    index = pl.read_csv(INDEX_PATH)
    print(index.select(pl.len().alias("files"), pl.col("participant_id").n_unique().alias("participants")))
else:
    index = None


## 3. GPU/CPU backend 확인


In [ ]:
# LightGBM은 device_type='cuda'와 device_type='gpu'(OpenCL)가 다릅니다.
# auto는 cuda를 먼저 probe하고, 안 되면 gpu, 그것도 안 되면 cpu를 선택합니다.
from llamac_research.device import select_lightgbm_device

backend = select_lightgbm_device(DEVICE)
print(f"요청한 DEVICE: {backend.requested}")
print(f"실제로 선택된 LightGBM device_type: {backend.selected}")
print(f"backend: {backend.backend}")
if backend.gpu_name:
    print(f"GPU 이름: {backend.gpu_name}")
if backend.fallback_reason:
    print(f"fallback 이유: {backend.fallback_reason}")


## 4. feature table 생성 또는 재사용


In [ ]:
# all = EEG/GSR/PPG/SKT/Respiration 전체 feature
# ppg = band_*.csv 안의 PPG feature만 사용
from llamac_research.features import build_feature_table, read_feature_table


def ensure_features(mode: str, path: Path) -> pl.DataFrame:
    """feature parquet이 있으면 읽고, 없으면 현재 설정으로 새로 만듭니다."""
    if path.exists() and not REBUILD_FEATURES:
        print(f"재사용: {path}")
        return read_feature_table(path)
    if SKIP_HEAVY:
        print(f"검증 모드라 feature 생성을 건너뜁니다: {path}")
        return pl.DataFrame()
    frame, summary = build_feature_table(
        EXTRACTED_DIR,
        mode=mode,
        limit_subjects=LIMIT_SUBJECTS,
        workers=WORKERS,
        output_path=path,
    )
    print(summary.to_dict())
    return frame

all_features = ensure_features("all", FEATURE_ALL)
ppg_features = ensure_features("ppg", FEATURE_PPG)

print("all:", all_features.shape)
print("ppg:", ppg_features.shape)
print("PPG feature 예시:", [c for c in ppg_features.columns if c.startswith("Band_PPG_")][:8])


## 5. LightGBM baseline 실행


In [ ]:
# grouped split은 참가자 단위로 train/test를 나눠 subject leakage를 막습니다.
# official-style stratified는 원본 notebook과 비슷한 비교용입니다.
import json

from llamac_research.modeling import ExperimentConfig, run_cross_validated_experiment, save_experiment_result
from llamac_research.metrics import metrics_summary_line


def run_or_load(label: str, config: ExperimentConfig, path: Path) -> dict | None:
    if path.exists() and not RERUN_EXPERIMENTS:
        print(f"[{label}] 저장된 결과 재사용: {path}")
        return json.loads(path.read_text())
    if SKIP_HEAVY:
        print(f"[{label}] 검증 모드라 학습을 건너뜁니다.")
        return None
    print(f"[{label}] 학습 시작")
    result = run_cross_validated_experiment(config)
    save_experiment_result(result, path)
    print(f"[{label}] {metrics_summary_line(result['metrics'])}")
    return result

experiments = {
    "all_grouped_reported": (
        ExperimentConfig(str(FEATURE_ALL), "lightgbm", "all", "reported", "grouped", FOLDS, SEED, DEVICE),
        RESULT_DIR / "all_grouped_reported.json",
    ),
    "ppg_grouped_reported": (
        ExperimentConfig(str(FEATURE_PPG), "lightgbm", "ppg", "reported", "grouped", FOLDS, SEED, DEVICE),
        RESULT_DIR / "ppg_grouped_reported.json",
    ),
    "all_official_intended": (
        ExperimentConfig(str(FEATURE_ALL), "lightgbm", "all", "intended", "stratified", FOLDS, SEED, DEVICE, True),
        RESULT_DIR / "all_official_intended.json",
    ),
}

results = {name: run_or_load(name, cfg, path) for name, (cfg, path) in experiments.items()}


## 6. 결과 요약과 간단한 그래프


In [ ]:
import matplotlib.pyplot as plt

summary_rows = []
for name, result in results.items():
    if result is None:
        continue
    m = result["metrics"]
    summary_rows.append({
        "run": name,
        "backend": result["backend"]["selected"],
        "rows": result["data"]["rows"],
        "features": result["data"]["candidate_features"],
        "top1": round(m["top1_accuracy"], 4),
        "top2": round(m["top2_accuracy"], 4),
        "top3": round(m["top3_accuracy"], 4),
        "macro_f1": round(m["macro_f1"], 4),
        "balanced_acc": round(m["balanced_accuracy"], 4),
        "kappa": round(m["cohen_kappa"], 4),
    })

summary = pl.DataFrame(summary_rows)
display(summary)

if summary.height:
    x = summary["run"].to_list()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x, summary["macro_f1"].to_numpy(), label="macro F1")
    ax.bar(x, summary["top1"].to_numpy(), alpha=0.55, label="top-1")
    ax.set_title("LightGBM baseline 요약")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=25)
    ax.legend()
    fig.tight_layout()
    plt.show()


## 7. PPG-only confusion matrix 확인


In [ ]:
# PPG-only가 어떤 감정을 헷갈리는지 빠르게 봅니다.
import numpy as np

EMOTION_LABELS = ["neutral", "fun", "sadness", "anger", "fear"]
ppg_result = results.get("ppg_grouped_reported")
if ppg_result is not None:
    cm = np.asarray(ppg_result["metrics"]["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(5), EMOTION_LABELS, rotation=45, ha="right")
    ax.set_yticks(range(5), EMOTION_LABELS)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title("PPG-only LightGBM confusion matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("PPG 결과가 아직 없습니다. 앞 셀에서 학습을 실행하세요.")
